## Passo a Passo Claude

### Step 1 — Point at the same catalog/schema/volume as Notebooks 02-03

In [0]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")
dbutils.widgets.text("volume_name", "synthetic_data", "Volume")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
documents_path = f"{volume_path}/documents"
pdf_volume_path = f"{documents_path}/16_fraud_escalation_playbook.pdf"

print(f"Expected PDF path: {pdf_volume_path}")

### Step 2 — Upload the pre-generated PDF to your Volume

1. In the Databricks workspace, go to **Catalog** (left sidebar) > your catalog > your schema > **Volumes** > your volume > the `documents/` folder.
2. Click **Upload to this volume** and select `pdfs/16_fraud_escalation_playbook.pdf` from this repo.
   - Alternatively, with the Databricks CLI configured locally: `databricks fs cp pdfs/16_fraud_escalation_playbook.pdf dbfs:/Volumes/<catalog>/<schema>/<volume>/documents/16_fraud_escalation_playbook.pdf`
3. Run the cell below to confirm it landed at the expected path -- it fails loudly with a clear message if not.

In [0]:
try:
    file_info = dbutils.fs.ls(pdf_volume_path)[0]
    print(f"Found: {file_info.path} ({file_info.size} bytes)")
except Exception as e:
    raise FileNotFoundError(
        f"Could not find {pdf_volume_path}. Upload pdfs/16_fraud_escalation_playbook.pdf from the "
        f"repo to this Volume path first -- see Step 2 above."
    ) from e

### Step 3 — Read it as binary, then parse it with `ai_parse_document`

This is the same `binaryFile` pattern from Notebook 03 -- `ai_parse_document` takes that raw binary `content` column as input. It doesn't matter that this file was uploaded rather than written by a previous cell; the bytes are the bytes.

In [0]:
pdf_binary_df = spark.read.format("binaryFile").load(pdf_volume_path)
pdf_binary_df.createOrReplaceTempView("fraud_pdf_raw")
display(pdf_binary_df.select("path", "length", "modificationTime"))

In [0]:
%sql
SELECT path, ai_parse_document(content) AS parsed
FROM fraud_pdf_raw

**Stop and look at the result above before writing any more code.** Expand the `parsed` column in the results grid -- that's the real, workspace-specific schema you'll work with. It will broadly contain something like a list of pages, each with elements tagged by type (heading, paragraph, table, ...), but the exact field names are a Databricks Runtime version detail we won't guess at here.

### Step 4 — Capture the result in Python and inspect its schema programmatically

Same query, but kept as a DataFrame so you can call `printSchema()` and look at one row directly -- this is the authoritative source of truth for field names in *your* workspace, not this notebook's prose.

In [0]:
parsed_df = spark.sql("SELECT path, ai_parse_document(content) AS parsed FROM fraud_pdf_raw")
parsed_df.printSchema()

parsed_row = parsed_df.first()
parsed_value = parsed_row["parsed"]
print(type(parsed_value))
print(parsed_value)

If `parsed_value` prints as a JSON string, the cell below pretty-prints it so it's easier to read and find the field names to use next. If it's already a native struct, `printSchema()` above already told you the field names to select.

In [0]:
import json

if isinstance(parsed_value, str):
    try:
        print(json.dumps(json.loads(parsed_value), indent=2))
    except json.JSONDecodeError:
        print("parsed_value is a string but not valid JSON -- inspect it as-is:")
        print(parsed_value)
else:
    print("parsed_value is already a structured type -- use parsed_df.printSchema() above to navigate it.")

## Explorando os campos e Explodindo gerando um Structed Dataframe

In [0]:
pdf_volume_path = "/Volumes/main/genai_lab/synthetic_data/documents/16_fraud_escalation_playbook.pdf"
pdf_binary_df = spark.read.format("binaryFile").load(pdf_volume_path)
pdf_binary_df.createOrReplaceTempView("fraud_pdf_raw")

In [0]:
%sql
WITH minha_tabela AS (
    SELECT
        path,
        ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM fraud_pdf_raw
)
SELECT
    path,
    pos AS element_position,
    value:confidence  AS confidence,
    value:content     AS content,
    value:description AS description,
    value:id           AS id,
    value:type         AS type
FROM minha_tabela,
LATERAL variant_explode(parsed:document:elements)

In [0]:
from pyspark.sql.functions import col, posexplode, expr

# Step 1: Parse the document using ai_parse_document (SQL function)
parsed_df = spark.sql("""
    SELECT path, ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM fraud_pdf_raw
""")

# Step 2: Explode the elements array with position, then extract VARIANT fields
# posexplode returns (pos, col) — equivalent to variant_explode on a VARIANT array
# try_cast converts the VARIANT array to ARRAY<VARIANT> so posexplode can iterate
# element:field accesses VARIANT sub-fields (same : syntax as SQL)
elements_df = (
    parsed_df
    .select(
        col("path"),
        posexplode(expr("try_cast(parsed:document:elements AS ARRAY<VARIANT>)")).alias("pos", "element")
    )
    .select(
        col("path"),
        col("pos").alias("element_position"),
        expr("element:confidence").alias("confidence"),
        expr("element:content").alias("content"),
        expr("element:description").alias("description"),
        expr("element:id").alias("id"),
        expr("element:type").alias("type"),
    )
)

display(elements_df)